# DTS304TC CW1: Insurance Premium Risk Classification

**Category:** B, C, E

---
## A Clean First Pipeline and Baseline Modelling

### A1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load pre-split datasets: 74,375 train / 13,125 val / 12,500 test (no labels)
train = pd.read_csv('dataset_final/train.csv')
val   = pd.read_csv('dataset_final/val.csv')
test  = pd.read_csv('dataset_final/test_features.csv')

print(f'Train shape : {train.shape}')
print(f'Val shape   : {val.shape}')
print(f'Test shape  : {test.shape}')

In [ ]:
# Preview raw rows to spot formatting issues and unexpected values
train.head(3)

In [ ]:
# Inspect dtypes to distinguish numeric features from categorical strings
train.dtypes

In [ ]:
# Check target class distribution to detect imbalance (Standard > High > Low)
print('Train label distribution:')
print(train['premium_risk'].value_counts())
print()
print('Val label distribution:')
print(val['premium_risk'].value_counts())

### A2. Identify and Remove Leakage & Admin Features

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

# Preserve admin IDs from test set for final submission output
test_admin = test[['applicant_id', 'customer_key']].copy()

# Step 1: compute Mutual Information (MI) between each numeric feature and the target
# MI measures non-linear statistical dependence; a score near 1.0 flags potential leakage
le = LabelEncoder()
y_train = le.fit_transform(train['premium_risk'])

# Fill NaNs with median before MI computation -> MI cannot handle missing values
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
X_num = train[numeric_cols].fillna(train[numeric_cols].median())

mi_scores = mutual_info_classif(X_num, y_train, random_state=42)
mi_df = pd.Series(mi_scores, index=numeric_cols).sort_values(ascending=False)

print('Mutual Information scores (numeric columns, vs premium_risk):')
print(mi_df.to_string())

In [ ]:
# Plot MI scores -> bureau_risk_index (MI ≈ 1.0) stands out as a clear leakage feature
fig, ax = plt.subplots(figsize=(9, 6))
mi_df.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Mutual Information with premium_risk')
ax.set_title('MI Scores | Numeric Features vs Target')
ax.invert_yaxis()

for bar, v in zip(ax.patches, mi_df.values):
    ax.text(
        v + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f'{v:.3f}',
        va='center', ha='left', fontsize=8
    )

ax.set_xlim(0, mi_df.max() * 1.18)
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: define columns to drop

# Admin / PII identifiers -> carry no predictive signal and must be excluded from modelling
ADMIN_COLS = ['applicant_id', 'customer_key', 'applicant_ref_code']

# Synthetic noise columns injected by the dataset provider -> confirmed near-zero MI
NOISE_COLS = [f'noise_feature_{i}' for i in range(1, 6)]

# Leakage feature: bureau_risk_index MI ≈ 1.0, meaning it directly encodes the target label
# Including it would make the model trivially accurate but useless in production
LEAK_COLS = ['bureau_risk_index']

DROP_COLS = ADMIN_COLS + NOISE_COLS + LEAK_COLS

print('Columns to drop:', DROP_COLS)

In [ ]:
def drop_cols(df, cols):
    # Silently skip columns absent from df (handles schema differences between train/val/test)
    existing = [c for c in cols if c in df.columns]
    return df.drop(columns=existing)

train_clean = drop_cols(train, DROP_COLS)
val_clean   = drop_cols(val,   DROP_COLS)
test_clean  = drop_cols(test,  DROP_COLS)

print('Remaining columns:', train_clean.columns.tolist())

### A3. Missing Values & Dirty Data

In [ ]:
# Compute missing rate per column; only columns with >0% missing are shown
missing = train_clean.isnull().mean().sort_values(ascending=False)
missing_report = missing[missing > 0]
print('Columns with missing values (train):')
print(missing_report.to_string() if len(missing_report) else 'None')

In [ ]:
# spending_profile contains garbled strings (e.g. '!@9#%8') -> replace with 'Unknown' category
# DIRTY_PATTERN matches any string composed entirely of non-alphanumeric characters
DIRTY_PATTERN = r'^[!@#\$%\^\&\*\(\)]+'

def clean_spending_profile(df):
    df = df.copy()
    if 'spending_profile' in df.columns:
        mask = df['spending_profile'].str.match(DIRTY_PATTERN, na=False)
        print(f'  Dirty spending_profile values replaced: {mask.sum()}')
        df.loc[mask, 'spending_profile'] = 'Unknown'
    return df

print('Train:')
train_clean = clean_spending_profile(train_clean)
print('Val:')
val_clean = clean_spending_profile(val_clean)
print('Test:')
test_clean = clean_spending_profile(test_clean)

In [ ]:
# account_tenure is stored as free text (e.g. '16 Years and 10 Months') -> parse to total months for numeric modelling
import re

def parse_tenure(s):
    if pd.isna(s):
        return np.nan
    years  = re.search(r'(\d+)\s*Year',  str(s))
    months = re.search(r'(\d+)\s*Month', str(s))
    y = int(years.group(1))  if years  else 0
    m = int(months.group(1)) if months else 0
    return y * 12 + m

for df in [train_clean, val_clean, test_clean]:
    df['account_tenure'] = df['account_tenure'].apply(parse_tenure)

print('account_tenure sample:', train_clean['account_tenure'].head(3).tolist())

In [ ]:
# Convert application_month from month name string to integer (1–12) for ordinal encoding
MONTH_MAP = {
    'January':1,'February':2,'March':3,'April':4,
    'May':5,'June':6,'July':7,'August':8,
    'September':9,'October':10,'November':11,'December':12
}

for df in [train_clean, val_clean, test_clean]:
    df['application_month'] = df['application_month'].map(MONTH_MAP)

print('application_month unique:', sorted(train_clean['application_month'].dropna().unique()))

In [ ]:
# prior_debt_products is a comma-separated product list -> convert to integer count
def count_debt_products(s):
    if pd.isna(s) or str(s).strip() == '':
        return 0
    return len(str(s).split(','))

for df in [train_clean, val_clean, test_clean]:
    df['prior_debt_products'] = df['prior_debt_products'].apply(count_debt_products)

print('prior_debt_products sample:', train_clean['prior_debt_products'].value_counts().head())

In [ ]:
# Binary encode minimum_payment_only: 'Yes' -> 1, 'No' -> 0
# NaN values are preserved for later imputation
for df in [train_clean, val_clean, test_clean]:
    df['minimum_payment_only'] = df['minimum_payment_only'].map({'Yes': 1, 'No': 0})

print('minimum_payment_only unique:', train_clean['minimum_payment_only'].unique())

In [ ]:
# Clip negative values in financial columns to 0 -> negative income or debt figures are physically impossible
num_cols_check = ['gross_annual_income_gbp', 'net_monthly_income_gbp',
                  'total_outstanding_debt_gbp', 'monthly_emi_total_gbp']

for col in num_cols_check:
    if col in train_clean.columns:
        neg = (train_clean[col] < 0).sum()
        if neg > 0:
            print(f'{col}: {neg} negative values -> clipped to 0')
            for df in [train_clean, val_clean, test_clean]:
                df[col] = df[col].clip(lower=0)

print('Outlier check complete.')

### A4. Encoding & Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Separate features and target; LABEL_ORDER defines ordinal rank (Low < Standard < High)
TARGET = 'premium_risk'
LABEL_ORDER = ['Low', 'Standard', 'High']

y_train = train_clean[TARGET]
y_val   = val_clean[TARGET]
X_train = train_clean.drop(columns=[TARGET])
X_val   = val_clean.drop(columns=[TARGET])
X_test  = test_clean.copy()

# Manually define categorical columns -> all remaining columns are treated as numeric
CAT_COLS = ['employment_sector', 'spending_profile', 'debt_portfolio_quality']
NUM_COLS = [c for c in X_train.columns if c not in CAT_COLS]

print(f'Numeric columns ({len(NUM_COLS)}): {NUM_COLS}')
print(f'Categorical columns ({len(CAT_COLS)}): {CAT_COLS}')

In [ ]:
# Numeric pipeline: median imputation (robust to outliers/skew) then z-score standardisation
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler())
])

# Categorical pipeline: mode imputation then OHE; handle_unknown='ignore' silences unseen categories at inference
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, NUM_COLS),
    ('cat', cat_pipe, CAT_COLS)
], remainder='drop')

# Fit only on train to prevent data leakage, val and test are transformed without re-fitting
X_train_pp = preprocessor.fit_transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

print(f'X_train_pp shape: {X_train_pp.shape}')
print(f'X_val_pp shape  : {X_val_pp.shape}')
print(f'X_test_pp shape : {X_test_pp.shape}')

### A5. Baseline Model (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

# Logistic Regression serves as the baseline: a linear model with no hyperparameter tuning
# max_iter=500 ensures convergence on this multi-class problem (default 100 is often insufficient)
baseline_lr = LogisticRegression(max_iter=500, random_state=42)
baseline_lr.fit(X_train_pp, y_train)

y_val_pred = baseline_lr.predict(X_val_pp)

acc_baseline = accuracy_score(y_val, y_val_pred)
f1_baseline  = f1_score(y_val, y_val_pred, average='macro')

y_tr_baseline   = baseline_lr.predict(X_train_pp)
acc_baseline_tr = accuracy_score(y_train, y_tr_baseline)
f1_baseline_tr  = f1_score(y_train, y_tr_baseline, average='macro')

print(f'Baseline LR | Train Acc: {acc_baseline_tr:.4f}, Train Macro-F1: {f1_baseline_tr:.4f}')
print(f'Baseline LR | Val   Acc: {acc_baseline:.4f},   Val   Macro-F1: {f1_baseline:.4f}')
print()
print(classification_report(y_val, y_val_pred, target_names=LABEL_ORDER))

In [ ]:
# Confusion matrix shows class-level errors, where 'Low' and 'Standard' are most confused by the baseline
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_val_pred,
    display_labels=LABEL_ORDER,
    colorbar=False,
    ax=ax
)
ax.set_title(f'Baseline LR | Val Confusion Matrix\nAcc={acc_baseline:.3f}, Macro-F1={f1_baseline:.3f}')
plt.tight_layout()
plt.show()

---
## B Controlled Comparison: Random Forest and One Boosting Model 

Both models use the **same preprocessing pipeline** fitted in Part A (`preprocessor`). Default / lightly-tuned parameters only; full hyperparameter search is deferred to Part C.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost requires integer class labels; RF accepts string labels natively
label_enc = LabelEncoder()
label_enc.fit(LABEL_ORDER)  # fixed order: High=0, Low=1, Standard=2

y_train_enc = label_enc.transform(y_train)
y_val_enc   = label_enc.transform(y_val)

### B1. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,     # enough trees for stable OOB; diminishing returns beyond ~500
    max_depth=None,       # fully-grown trees — bagging relies on high-variance base learners
    min_samples_leaf=2,   # minimal regularisation to avoid pure singleton leaves
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_pp, y_train)

y_val_rf  = rf.predict(X_val_pp)
acc_rf    = accuracy_score(y_val, y_val_rf)
f1_rf     = f1_score(y_val, y_val_rf, average='macro')

# train metrics, compare with val to quantify the overfitting gap
y_tr_rf   = rf.predict(X_train_pp)
acc_rf_tr = accuracy_score(y_train, y_tr_rf)
f1_rf_tr  = f1_score(y_train, y_tr_rf, average='macro')

print(f'Random Forest | Train Acc: {acc_rf_tr:.4f}, Train Macro-F1: {f1_rf_tr:.4f}')
print(f'Random Forest | Val   Acc: {acc_rf:.4f}, Val   Macro-F1: {f1_rf:.4f}')
print()
print(classification_report(y_val, y_val_rf, target_names=LABEL_ORDER))

### B2. XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,         # boosting rounds; matches RF for a fair comparison
    max_depth=6,              # XGBoost default; shallower than RF's unbounded depth
    learning_rate=0.1,        # classic starting point; shrinks each tree's contribution
    subsample=0.8,            # row subsampling per round; reduces overfitting
    colsample_bytree=0.8,     # feature subsampling per tree; analogous to RF's max_features
    eval_metric='mlogloss',   # multi-class log loss for internal monitoring
    n_jobs=-1,
    random_state=42
)
xgb.fit(X_train_pp, y_train_enc)

y_val_xgb_enc = xgb.predict(X_val_pp)
y_val_xgb     = label_enc.inverse_transform(y_val_xgb_enc) # decode integers back to class strings

acc_xgb = accuracy_score(y_val, y_val_xgb)
f1_xgb  = f1_score(y_val, y_val_xgb, average='macro')

# train metrics, compare with val to quantify the overfitting gap
y_tr_xgb_enc = xgb.predict(X_train_pp)
y_tr_xgb     = label_enc.inverse_transform(y_tr_xgb_enc)
acc_xgb_tr   = accuracy_score(y_train, y_tr_xgb)
f1_xgb_tr    = f1_score(y_train, y_tr_xgb, average='macro')

print(f'XGBoost | Train Acc: {acc_xgb_tr:.4f}, Train Macro-F1: {f1_xgb_tr:.4f}')
print(f'XGBoost | Val   Acc: {acc_xgb:.4f}, Val   Macro-F1: {f1_xgb:.4f}')
print()
print(classification_report(y_val, y_val_xgb, target_names=LABEL_ORDER))

### B3. Comparison

In [ ]:
# aggregate train/val metrics across all three models for direct comparison
comparison = pd.DataFrame({
    'Model':          ['Baseline LR',        'Random Forest',    'XGBoost'],
    'Train Acc':      ['-',                   f'{acc_rf_tr:.4f}', f'{acc_xgb_tr:.4f}'],
    'Val Acc':        [f'{acc_baseline:.4f}', f'{acc_rf:.4f}',    f'{acc_xgb:.4f}'],
    'Train Macro-F1': ['-',                   f'{f1_rf_tr:.4f}',  f'{f1_xgb_tr:.4f}'],
    'Val Macro-F1':   [f'{f1_baseline:.4f}',  f'{f1_rf:.4f}',     f'{f1_xgb:.4f}'],
})
print(comparison.to_string(index=False))

In [ ]:
# Side-by-side confusion matrices -> reveals per-class error patterns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_val, y_val_rf,
    display_labels=LABEL_ORDER, colorbar=False, ax=axes[0]
)
axes[0].set_title(f'Random Forest\nAcc={acc_rf:.3f}, Macro-F1={f1_rf:.3f}')

ConfusionMatrixDisplay.from_predictions(
    y_val, y_val_xgb,
    display_labels=LABEL_ORDER, colorbar=False, ax=axes[1]
)
axes[1].set_title(f'XGBoost\nAcc={acc_xgb:.3f}, Macro-F1={f1_xgb:.3f}')

plt.tight_layout()
plt.show()

In [ ]:
# Recover feature names after OHE expansion
ohe_cats     = preprocessor.named_transformers_['cat']['ohe'].get_feature_names_out(CAT_COLS)
feature_names = NUM_COLS + list(ohe_cats)

rf_imp  = pd.Series(rf.feature_importances_,  index=feature_names)
xgb_imp = pd.Series(xgb.feature_importances_, index=feature_names)

top_n       = 15
top_feats   = rf_imp.nlargest(top_n).index  # RF's top-N used as shared reference set for both plots

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rf_imp[top_feats].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title(f'RF Top {top_n} Feature Importances')
axes[0].set_xlabel('Importance')

xgb_imp[top_feats].sort_values().plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title(f'XGBoost Top {top_n} Feature Importances (same features)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

### B4. Interpretation: Bagging vs Boosting on This Dataset

**Algorithmic difference.**  
Random Forest (bagging) trains trees in parallel on bootstrap samples and averages their predictions; variance is reduced but bias is unchanged. XGBoost (boosting) fits each new tree to the residual errors of the current ensemble, correcting both bias and variance through sequential updates with regularisation.

**Evidence from this dataset.**

| Model | Train Macro-F1 | Val Macro-F1 | Gap |
|-------|---------------|-------------|-----|
| Random Forest | 0.9854 | 0.7985 | **0.187** |
| XGBoost | 0.8952 | 0.8384 | 0.057 |

Random Forest overfits severely (gap = 0.187). XGBoost generalises far better (gap = 0.057) due to its built-in L1/L2 regularisation and learning-rate shrinkage.

**Conclusion.**  
XGBoost is selected for hyperparameter optimisation in Part C.

---
## C Advanced Hyperparameter Optimisation

In [ ]:
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'eval_metric': 'mlogloss',
        'n_jobs': -1,
        'random_state': 42,
        'device': 'cuda',
    }
    model = XGBClassifier(**params)
    model.fit(X_train_pp, y_train_enc)
    preds = model.predict(X_val_pp)
    preds_str = label_enc.inverse_transform(preds)
    return f1_score(y_val, preds_str, average='macro')

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f'\nBest Val Macro-F1 : {study.best_value:.4f}')
print(f'Best params       : {study.best_params}')

In [ ]:
best_params = study.best_params
best_params.update({
    'device': 'cuda',
    'eval_metric': 'mlogloss',
    'n_jobs': -1,
    'random_state': 42
})

xgb_opt = XGBClassifier(**best_params)
xgb_opt.fit(X_train_pp, y_train_enc)

y_val_opt_enc = xgb_opt.predict(X_val_pp)
y_val_opt     = label_enc.inverse_transform(y_val_opt_enc)

acc_opt = accuracy_score(y_val, y_val_opt)
f1_opt  = f1_score(y_val, y_val_opt, average='macro')

y_tr_opt_enc = xgb_opt.predict(X_train_pp)
y_tr_opt     = label_enc.inverse_transform(y_tr_opt_enc)
acc_opt_tr   = accuracy_score(y_train, y_tr_opt)
f1_opt_tr    = f1_score(y_train, y_tr_opt, average='macro')

print(f'XGBoost (Optuna) | Train Acc: {acc_opt_tr:.4f}, Train Macro-F1: {f1_opt_tr:.4f}')
print(f'XGBoost (Optuna) | Val   Acc: {acc_opt:.4f},   Val   Macro-F1: {f1_opt:.4f}')
print()
print(classification_report(y_val, y_val_opt, target_names=LABEL_ORDER))

In [ ]:
# Optimisation history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Trial-by-trial F1
trials_df = study.trials_dataframe()
axes[0].plot(trials_df['number'], trials_df['value'], alpha=0.4, color='gray', label='Each trial')
axes[0].plot(trials_df['number'],
             trials_df['value'].cummax(),
             color='steelblue', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Val Macro-F1')
axes[0].set_title('Optuna Optimisation History')
axes[0].legend()

# Hyperparameter importance
importances = optuna.importance.get_param_importances(study)
imp_series  = pd.Series(importances).sort_values()
imp_series.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Hyperparameter Importance')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame({
    'Model':        ['Baseline LR', 'Random Forest', 'XGBoost (default)', 'XGBoost (Optuna)'],
    'Val Acc':      [f'{acc_baseline:.4f}', f'{acc_rf:.4f}', f'{acc_xgb:.4f}', f'{acc_opt:.4f}'],
    'Val Macro-F1': [f'{f1_baseline:.4f}', f'{f1_rf:.4f}',  f'{f1_xgb:.4f}',  f'{f1_opt:.4f}'],
})
print(summary.to_string(index=False))

### Search Space and Optimiser Choice

**Optimiser:** Optuna TPE (Tree-structured Parzen Estimator) — a Bayesian sequential optimiser that prioritises promising hyperparameter regions, more sample-efficient than grid or random search over a continuous 8-dimensional space across 100 trials.

**Search space:**

| Parameter | Range |
|-----------|-------|
| `n_estimators` | 100 – 1000 |
| `max_depth` | 3 – 10 |
| `learning_rate` | 0.01 – 0.3 (log) |
| `subsample` | 0.6 – 1.0 |
| `colsample_bytree` | 0.6 – 1.0 |
| `min_child_weight` | 1 – 10 |
| `reg_alpha` | 1e-8 – 10 (log) |
| `reg_lambda` | 1e-8 – 10 (log) |

**Best trial:** Val Macro-F1 = **0.8843** (`n_estimators=727`, `max_depth=9`, `learning_rate=0.189`, `subsample=0.979`, `colsample_bytree=0.677`, `min_child_weight=1`, `reg_alpha=0.205`, `reg_lambda=0.039`), +0.046 over default XGBoost. `learning_rate`, `max_depth`, and `n_estimators` contributed most to performance variance (see importance plot above).

---
## D Personalised Improvement Work

**Compulsory:** Category B Feature Representation and Engineering  (assigned by student-ID digit)  
**Optional 1:** Category C Imbalance and Objective Design  
**Optional 2:** Category E Fairness, Diagnostics, or Interpretability 


### Category B: Feature Representation and Engineering
Ablation over four feature configurations, all using the Optuna-tuned XGBoost hyperparameters from Part C.

In [ ]:
# Feature engineering helpers
LOG_COLS = [c for c in [
    'gross_annual_income_gbp', 'net_monthly_income_gbp',
    'total_outstanding_debt_gbp', 'monthly_emi_total_gbp',
    'avg_payment_delay_days', 'credit_inquiry_count'
] if c in X_train.columns]

INT_COLS = ['debt_to_income', 'emi_to_income', 'late_x_utilisation', 'income_per_card']

def add_log_transforms(df):
    df = df.copy()
    for col in LOG_COLS:
        df[f'log_{col}'] = np.log1p(df[col].clip(lower=0).fillna(0))
    return df

def add_interactions(df):
    df = df.copy()
    eps = 1e-6
    df['debt_to_income'] = df['total_outstanding_debt_gbp'].fillna(0) / df['gross_annual_income_gbp'].clip(lower=eps).fillna(eps)
    df['emi_to_income'] = df['monthly_emi_total_gbp'].fillna(0) / df['net_monthly_income_gbp'].clip(lower=eps).fillna(eps)
    df['late_x_utilisation'] = df['late_payment_count'].fillna(0) * df['credit_utilisation_pct'].fillna(0)
    df['income_per_card'] = df['gross_annual_income_gbp'].fillna(0) / df['credit_cards_held'].clip(lower=eps).fillna(eps)
    return df

class SmoothTargetEncoder:
    """Smoothed mean-target encoder; fit on train only to prevent leakage."""
    def __init__(self, cols, smoothing=10):
        self.cols, self.smoothing = cols, smoothing
        self.encodings_, self.global_mean_ = {}, None

    def fit(self, X, y_numeric):
        self.global_mean_ = float(np.nanmean(y_numeric))
        for col in self.cols:
            tmp = pd.DataFrame({'col': X[col].values, 'y': y_numeric})
            stats = tmp.groupby('col')['y'].agg(['mean', 'count'])
            s = self.smoothing / (self.smoothing + stats['count'])
            self.encodings_[col] = (stats['mean'] * (1 - s) + self.global_mean_ * s).to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = X[col].map(self.encodings_[col]).fillna(self.global_mean_)
        return X

# Ordinal mapping for target encoder (Low < Standard < High)
RISK_ORDINAL = {'Low': 0, 'Standard': 1, 'High': 2}
y_train_ordinal = y_train.map(RISK_ORDINAL).values

In [ ]:
def build_and_eval(name, X_tr, X_vl, num_c, cat_c):
    transformers = [('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), num_c)]
    if cat_c:
        transformers.append(('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_c))

    pp  = ColumnTransformer(transformers, remainder='drop')
    Xtr = pp.fit_transform(X_tr)
    Xvl = pp.transform(X_vl)

    mdl = XGBClassifier(**best_params)
    mdl.fit(Xtr, y_train_enc)

    pred = label_enc.inverse_transform(mdl.predict(Xvl))
    acc  = accuracy_score(y_val, pred)
    f1   = f1_score(y_val, pred, average='macro')
    f1s  = f1_score(y_val, pred, average=None, labels=LABEL_ORDER)

    print(f'[{name}]  Acc={acc:.4f}  Macro-F1={f1:.4f}  |  '
          f'F1-Low={f1s[0]:.4f}  F1-Std={f1s[1]:.4f}  F1-High={f1s[2]:.4f}')
    return dict(name=name, acc=acc, f1=f1,
                f1_low=f1s[0], f1_std=f1s[1], f1_high=f1s[2],
                pp=pp, mdl=mdl, Xtr=Xtr, Xvl=Xvl)

log_new = [f'log_{c}' for c in LOG_COLS]

# B0: Part C baseline
r_b0 = build_and_eval('B0 Baseline', X_train, X_val, NUM_COLS, CAT_COLS)

# B1: + log transforms
X_tr1 = add_log_transforms(X_train)
X_vl1 = add_log_transforms(X_val)
r_b1 = build_and_eval('B1 + Log', X_tr1, X_vl1, NUM_COLS + log_new, CAT_COLS)

# B2: + interaction features
X_tr2 = add_interactions(X_tr1)
X_vl2 = add_interactions(X_vl1)
r_b2 = build_and_eval('B2 + Log + Inter', X_tr2, X_vl2, NUM_COLS + log_new + INT_COLS, CAT_COLS)

# B3: + target encoding (OHE replaced)
ste  = SmoothTargetEncoder(cols=CAT_COLS, smoothing=10).fit(X_train, y_train_ordinal)
X_tr3 = ste.transform(X_tr2)
X_vl3 = ste.transform(X_vl2)
r_b3 = build_and_eval('B3 + Log + Inter + TE', X_tr3, X_vl3,
                       NUM_COLS + log_new + INT_COLS + CAT_COLS, [])

In [ ]:
# Category B ablation results
ablation_b = pd.DataFrame([
    {'Config': r['name'], 'Val Acc': f"{r['acc']:.4f}", 'Macro-F1': f"{r['f1']:.4f}",
     'F1 Low': f"{r['f1_low']:.4f}", 'F1 Standard': f"{r['f1_std']:.4f}", 'F1 High': f"{r['f1_high']:.4f}"}
    for r in [r_b0, r_b1, r_b2, r_b3]
])
print('Category B Ablation Table')
print(ablation_b.to_string(index=False))

best_b = max([r_b0, r_b1, r_b2, r_b3], key=lambda r: r['f1'])
print(f"\nBest B config: '{best_b['name']}'  "
      f"Val Acc={best_b['acc']:.4f}  Macro-F1={best_b['f1']:.4f}")

### Category B Interpretation

| Config | Macro-F1 | Delta |
|--------|----------|-------|
| B0 Baseline | 0.8843 | — |
| B1 + Log | 0.8804 | -0.0039 |
| B2 + Log + Inter | 0.8898 | +0.0094 |
| B3 + Log + Inter + TE | 0.8913 | +0.0015 |

Interaction features (B2) gave the largest gain; target encoding (B3) added a further marginal improvement. Log transforms alone slightly hurt performance.

**Best config: B3** (Val Acc = 0.9035, Macro-F1 = 0.8913), carried forward to Part D.

### Category C: Imbalance and Objective Design
Three strategies applied on top of the best feature config from Category B.
Class distribution: Standard (53%) > High (29%) > Low (18%); Low is the minority class.

In [ ]:
# C1: Class-weighted XGBoost
from sklearn.utils.class_weight import compute_sample_weight

# Reuse best-B preprocessed arrays
X_tr_c = best_b['Xtr']
X_vl_c = best_b['Xvl']

print('Class distribution (train):')
for cls, cnt in zip(*np.unique(y_train, return_counts=True)):
    print(f'  {cls:10s}: {cnt:6d} ({cnt/len(y_train)*100:.1f}%)')

sw = compute_sample_weight('balanced', y_train_enc)
xgb_cw = XGBClassifier(**best_params)
xgb_cw.fit(X_tr_c, y_train_enc, sample_weight=sw)

pred_cw = label_enc.inverse_transform(xgb_cw.predict(X_vl_c))
acc_cw = accuracy_score(y_val, pred_cw)
f1_cw = f1_score(y_val, pred_cw, average='macro')
f1s_cw = f1_score(y_val, pred_cw, average=None, labels=LABEL_ORDER)

print(f'\nC1 Class-weighted  Acc={acc_cw:.4f}  Macro-F1={f1_cw:.4f}')
print(classification_report(y_val, pred_cw, target_names=LABEL_ORDER))

In [ ]:
# C2: SMOTE
from imblearn.over_sampling import SMOTE
from collections import Counter

sm = SMOTE(random_state=42, k_neighbors=5)
X_sm, y_sm = sm.fit_resample(X_tr_c, y_train_enc)

before = {label_enc.inverse_transform([k])[0]: v for k, v in Counter(y_train_enc).items()}
after  = {label_enc.inverse_transform([k])[0]: v for k, v in Counter(y_sm).items()}
print(f'Before SMOTE: {before}')
print(f'After SMOTE:  {after} (total: {len(y_sm)})')

xgb_sm = XGBClassifier(**best_params)
xgb_sm.fit(X_sm, y_sm)

pred_sm = label_enc.inverse_transform(xgb_sm.predict(X_vl_c))
acc_sm  = accuracy_score(y_val, pred_sm)
f1_sm   = f1_score(y_val, pred_sm, average='macro')
f1s_sm  = f1_score(y_val, pred_sm, average=None, labels=LABEL_ORDER)

print(f'\nC2 SMOTE  Acc={acc_sm:.4f}  Macro-F1={f1_sm:.4f}')
print(classification_report(y_val, pred_sm, target_names=LABEL_ORDER))

In [ ]:
# C3: probability threshold tuning (class-weighted model)
from scipy.optimize import minimize

proba_vl = xgb_cw.predict_proba(X_vl_c)   # shape (N_val, 3)

def neg_f1(t):
    scaled = proba_vl * np.clip(t, 1e-9, None)
    pred = label_enc.inverse_transform(np.argmax(scaled, axis=1))
    return -f1_score(y_val, pred, average='macro')

res = minimize(neg_f1, [1.0, 1.0, 1.0], method='Nelder-Mead',
                 options={'maxiter': 1000, 'xatol': 1e-5, 'fatol': 1e-5})
t_opt = res.x / res.x.sum()

print('Optimised thresholds:')
for cls, t in zip(label_enc.classes_, t_opt):
    print(f' {cls}: {t:.4f}')

pred_t = label_enc.inverse_transform(np.argmax(proba_vl * t_opt, axis=1))
acc_t  = accuracy_score(y_val, pred_t)
f1_t   = f1_score(y_val, pred_t, average='macro')
f1s_t  = f1_score(y_val, pred_t, average=None, labels=LABEL_ORDER)

print(f'\nC3 Threshold-tuned  Acc={acc_t:.4f}  Macro-F1={f1_t:.4f}')
print(classification_report(y_val, pred_t, target_names=LABEL_ORDER))

In [ ]:
# Category C ablation results
ablation_c = pd.DataFrame([
    {'Config': name, 'Val Acc': f'{acc:.4f}', 'Macro-F1': f'{f1:.4f}',
     'F1 Low': f'{f1s[0]:.4f}', 'F1 Standard': f'{f1s[1]:.4f}', 'F1 High': f'{f1s[2]:.4f}'}
    for name, acc, f1, f1s in [
        ('C0 Base (best-B)',  best_b['acc'], best_b['f1'],
         [best_b['f1_low'], best_b['f1_std'], best_b['f1_high']]),
        ('C1 Class-weighted', acc_cw,  f1_cw,  f1s_cw),
        ('C2 SMOTE',          acc_sm,  f1_sm,  f1s_sm),
        ('C3 Thresh-tuned',   acc_t,   f1_t,   f1s_t),
    ]
])
print('Category C Ablation Table')
print(ablation_c.to_string(index=False))

# Best model carried forward to Part F
_candidates = [
    dict(name='best-B',       acc=best_b['acc'], f1=best_b['f1'], mdl=best_b['mdl'], pp=best_b['pp']),
    dict(name='C1-classweight', acc=acc_cw,  f1=f1_cw,  mdl=xgb_cw, pp=best_b['pp']),
    dict(name='C2-SMOTE',       acc=acc_sm,  f1=f1_sm,  mdl=xgb_sm, pp=best_b['pp']),
]
best_d = max(_candidates, key=lambda r: r['f1'])
print(f"\nBest overall Part D model: '{best_d['name']}'  "
      f"Val Acc={best_d['acc']:.4f}  Macro-F1={best_d['f1']:.4f}")

### Category C Interpretation

| Strategy | F1-Low | F1-Standard | F1-High | Macro-F1 |
|----------|--------|-------------|---------|----------|
| C0 Base (best-B) | 0.8293 | 0.9128 | 0.9318 | 0.8913 |
| C1 Class-weighted | **0.8306** | 0.9122 | 0.9304 | 0.8911 |
| C2 SMOTE | 0.8160 | **0.9153** | **0.9383** | 0.8899 |
| C3 Thresh-tuned | **0.8311** | 0.9129 | 0.9311 | **0.8917** |

C1 and C3 both marginally improve F1-Low; C2 (SMOTE) unexpectedly hurts it. Overall gains are modest as the tuned XGBoost already handles imbalance implicitly.

**Best config: C3** (Macro-F1 = 0.8917, F1-Low = 0.8311).

### Category E: Fairness, Diagnostics and Interpretability
SHAP analysis on the Optuna-tuned XGBoost (Part C); same model, original preprocessing, clean feature names.
Followed by error analysis focusing on the Standard class (lowest F1).

In [ ]:
# SHAP calculation
import shap

ohe_feats  = preprocessor.named_transformers_['cat']['ohe'].get_feature_names_out(CAT_COLS)
feat_names = NUM_COLS + list(ohe_feats)
print(f'Total features: {len(feat_names)}')

# Explainer on xgb_opt (Part C model)
explainer = shap.TreeExplainer(xgb_opt)

# Subsample 2000 val observations for speed
rng   = np.random.RandomState(42)
idx_s = rng.choice(len(X_val_pp), 2000, replace=False)
X_shap = X_val_pp[idx_s]
y_shap = np.array(y_val.values)[idx_s]

print('Computing SHAP values...')
shap_vals = explainer.shap_values(X_shap)

if not isinstance(shap_vals, list):
    shap_vals = [shap_vals[:, :, i] for i in range(shap_vals.shape[2])]

# shap_vals: list of 3 arrays (one per class in label_enc.classes_ order)
print(f'Classes: {list(label_enc.classes_)}')
print(f'Per-class SHAP array shape: {shap_vals[0].shape}')

In [ ]:
# Global SHAP importance: mean |SHAP| across all classes
mean_abs = np.mean([np.abs(sv) for sv in shap_vals], axis=0).mean(axis=0)

top_k     = 15
top_idx   = np.argsort(mean_abs)[-top_k:][::-1]
top_names = [feat_names[i] for i in top_idx]
top_vals  = mean_abs[top_idx]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(top_k), top_vals[::-1], color='steelblue')
ax.set_yticks(range(top_k))
ax.set_yticklabels(top_names[::-1])
ax.set_xlabel('Mean |SHAP value| (averaged across classes)')
ax.set_title(f'Top {top_k} Features | Global SHAP Importance')
plt.tight_layout()
plt.show()

print('\nTop 10 global features:')
for name, val in zip(top_names[:10], top_vals[:10]):
    print(f'  {name:45s}: {val:.4f}')

In [ ]:
# Per-class SHAP beeswarm (top 10 features)
top10_idx   = top_idx[:10]
top10_names = [feat_names[i] for i in top10_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, cls_name in enumerate(label_enc.classes_):
    plt.sca(axes[i])
    shap.summary_plot(
        shap_vals[i][:, top10_idx],
        X_shap[:, top10_idx],
        feature_names=top10_names,
        show=False, plot_size=None
    )
    axes[i].set_title(f'SHAP Beeswarm | Class: {cls_name}')

plt.tight_layout()
plt.show()

In [ ]:
# Error Analysis
y_pred_e    = label_enc.inverse_transform(xgb_opt.predict(X_val_pp))
err_mask    = np.array(y_val.values) != y_pred_e

print(f'Total misclassified: {err_mask.sum()} / {len(y_val)} ({err_mask.mean():.2%})\n')

print('Error rate by true class:')
for cls in LABEL_ORDER:
    m   = np.array(y_val.values) == cls
    n   = (err_mask & m).sum()
    tot = m.sum()
    print(f'  {cls:10s}: {n}/{tot} ({n/tot:.2%})')

print('\nTop confusion pairs (true → predicted):')
err_df = pd.DataFrame({'true': y_val.values, 'pred': y_pred_e})
print(err_df[err_df['true'] != err_df['pred']]
        .groupby(['true','pred']).size()
        .sort_values(ascending=False).head(6).to_string())

In [ ]:
# Standard class error analysis: SHAP difference (misclassified vs correct)
std_enc  = label_enc.transform(['Standard'])[0]
shap_std = shap_vals[std_enc]   # shape (2000, n_feats)

y_shap_str  = y_shap
pred_sample = label_enc.inverse_transform(xgb_opt.predict(X_shap))

std_err_mask = (y_shap_str == 'Standard') & (pred_sample != 'Standard')
std_ok_mask  = (y_shap_str == 'Standard') & (pred_sample == 'Standard')

print(f'Standard in sample: correct: {std_ok_mask.sum()}, wrong: {std_err_mask.sum()}')

if std_err_mask.sum() > 0 and std_ok_mask.sum() > 0:
    diff = (np.abs(shap_std[std_err_mask]).mean(axis=0) -
            np.abs(shap_std[std_ok_mask]).mean(axis=0))
    top_diff = np.argsort(np.abs(diff))[-10:][::-1]

    print('\nFeatures with largest SHAP difference (misclassified vs correct Standard):')
    print(f'{"Feature":45s}  {"Wrong":>8}  {"Correct":>8}  {"Delta":>8}')
    for j in top_diff:
        w = np.abs(shap_std[std_err_mask, j]).mean()
        c = np.abs(shap_std[std_ok_mask,  j]).mean()
        print(f'{feat_names[j]:45s}  {w:8.4f}  {c:8.4f}  {w-c:+8.4f}')

### Category E Interpretation

**Global SHAP importance:** top features are `late_payment_count`, `total_outstanding_debt_gbp`, `monthly_emi_total_gbp`, and `gross_annual_income_gbp`. Noise features show near-zero SHAP values, confirming correct exclusion.

**Error analysis:** 1,351 / 13,125 val samples misclassified (10.3%).

| Class | Error rate |
|-------|-----------|
| Low | 17.3% |
| High | 10.8% |
| Standard | 7.7% |

Dominant confusion: **Low ↔ Standard** (397 + 394 errors).

**SHAP difference (Standard errors):** misclassified Standard samples show lower |SHAP| on `late_payment_count` and `total_outstanding_debt_gbp`, indicating the model loses confidence when payment-behaviour signals are weak.

In [ ]:
# Part D ablation summary
summary_d = pd.DataFrame([
    {'Part': 'C (baseline)',  'Config': 'C0 Base (best-B)',   'Val Acc': f"{best_b['acc']:.4f}", 'Macro-F1': f"{best_b['f1']:.4f}"},
    {'Part': 'D-B',          'Config': r_b0['name'],         'Val Acc': f"{r_b0['acc']:.4f}",   'Macro-F1': f"{r_b0['f1']:.4f}"},
    {'Part': 'D-B',          'Config': r_b1['name'],         'Val Acc': f"{r_b1['acc']:.4f}",   'Macro-F1': f"{r_b1['f1']:.4f}"},
    {'Part': 'D-B',          'Config': r_b2['name'],         'Val Acc': f"{r_b2['acc']:.4f}",   'Macro-F1': f"{r_b2['f1']:.4f}"},
    {'Part': 'D-B',          'Config': r_b3['name'],         'Val Acc': f"{r_b3['acc']:.4f}",   'Macro-F1': f"{r_b3['f1']:.4f}"},
    {'Part': 'D-C',          'Config': 'C1 Class-weighted',  'Val Acc': f'{acc_cw:.4f}',        'Macro-F1': f'{f1_cw:.4f}'},
    {'Part': 'D-C',          'Config': 'C2 SMOTE',           'Val Acc': f'{acc_sm:.4f}',        'Macro-F1': f'{f1_sm:.4f}'},
    {'Part': 'D-C',          'Config': 'C3 Thresh-tuned',    'Val Acc': f'{acc_t:.4f}',         'Macro-F1': f'{f1_t:.4f}'},
])
print('Part D Complete Ablation Summary')
print(summary_d.to_string(index=False))
print(f"\nFinal best model for Part F: '{best_d['name']}'  Macro-F1={best_d['f1']:.4f}")

---
## E K-Means and Gaussian Mixture Model (GMM) Exploration


In [ ]:
# Feature space setup (PCA)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# 10 components: compact representation, speeds up clustering
pca   = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_val_pp)

print(f'Feature space: X_val_pp -> PCA(10)')
print(f'Explained variance ratio (10 PCs): {pca.explained_variance_ratio_.sum():.3f}')
print(f'Shape: {X_pca.shape}')

# Subsample 5000 points for GMM speed
rng_e = np.random.RandomState(0)
idx_e = rng_e.choice(len(X_pca), 5000, replace=False)
X_e   = X_pca[idx_e]
y_e   = np.array(y_val.values)[idx_e]   # true labels, reference only

In [ ]:
# K-Means exploration
K_RANGE = range(2, 9)

km_inertia    = []
km_silhouette = []
km_labels_all = {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_e)
    km_inertia.append(km.inertia_)
    sil = silhouette_score(X_e, labels, sample_size=2000, random_state=42)
    km_silhouette.append(sil)
    km_labels_all[k] = labels

# Cluster sizes for k=3 (matches number of classes)
k_show = 3
sizes_km = pd.Series(km_labels_all[k_show]).value_counts().sort_index()
print(f'K-Means (k={k_show}) cluster sizes:')
print(sizes_km.to_string())

# Cross-tab with true labels (reference only)
cross_km = pd.crosstab(
    pd.Series(km_labels_all[k_show], name='KM Cluster'),
    pd.Series(y_e, name='True Label')
)
print(f'\nCross-tab (k={k_show}) | clusters vs true labels (reference only):')
print(cross_km)

In [ ]:
# GMM exploration
gmm_bic        = []
gmm_aic        = []
gmm_loglik     = []
gmm_labels_all = {}

for k in K_RANGE:
    gm = GaussianMixture(n_components=k, covariance_type='full',
                         max_iter=200, random_state=42)
    gm.fit(X_e)
    gmm_bic.append(gm.bic(X_e))
    gmm_aic.append(gm.aic(X_e))
    gmm_loglik.append(gm.score(X_e) * len(X_e))   # total log-likelihood
    gmm_labels_all[k] = gm.predict(X_e)

# Component sizes and posterior uncertainty for k=3
k_show = 3
gm3 = GaussianMixture(n_components=k_show, covariance_type='full',
                      max_iter=200, random_state=42)
gm3.fit(X_e)

proba3  = gm3.predict_proba(X_e)           # shape (N, 3)
max_p   = proba3.max(axis=1)               # confidence of best component
entropy = -(proba3 * np.log(proba3 + 1e-12)).sum(axis=1)  # uncertainty

print(f'GMM (k={k_show}) component weights: {gm3.weights_.round(3)}')
print(f'Component sizes: {np.bincount(gm3.predict(X_e))}')
print(f'\nPosterior confidence (max responsibility):')
print(f'  Mean : {max_p.mean():.3f}')
print(f'  Median: {np.median(max_p):.3f}')
print(f'  % with confidence > 0.9: {(max_p > 0.9).mean():.2%}')
print(f'\nPosterior entropy (uncertainty):')
print(f'  Mean : {entropy.mean():.3f}  (0=certain, log(k)={np.log(k_show):.3f}=max uncertain)')

# Cross-tab with true labels
cross_gmm = pd.crosstab(
    pd.Series(gm3.predict(X_e), name='GMM Component'),
    pd.Series(y_e, name='True Label')
)
print(f'\nCross-tab (k={k_show}) | GMM components vs true labels (reference only):')
print(cross_gmm)

In [ ]:
# Comparison figure and table
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

ks = list(K_RANGE)

# (0,0) K-Means elbow
axes[0,0].plot(ks, km_inertia, 'o-', color='steelblue')
axes[0,0].set_xlabel('k')
axes[0,0].set_ylabel('Inertia (SSE)')
axes[0,0].set_title('K-Means | Elbow (SSE)')
axes[0,0].set_xticks(ks)

# (0,1) K-Means silhouette
axes[0,1].plot(ks, km_silhouette, 's-', color='steelblue')
axes[0,1].set_xlabel('k')
axes[0,1].set_ylabel('Silhouette Score')
axes[0,1].set_title('K-Means | Silhouette Score')
axes[0,1].set_xticks(ks)

# (0,2) GMM BIC and AIC
axes[0,2].plot(ks, gmm_bic, 'o-', label='BIC', color='darkorange')
axes[0,2].plot(ks, gmm_aic, 's--', label='AIC', color='tomato')
axes[0,2].set_xlabel('k')
axes[0,2].set_ylabel('Score (lower = better)')
axes[0,2].set_title('GMM | BIC & AIC')
axes[0,2].legend()
axes[0,2].set_xticks(ks)

# (1,0) GMM log-likelihood
axes[1,0].plot(ks, gmm_loglik, 'D-', color='darkorange')
axes[1,0].set_xlabel('k')
axes[1,0].set_ylabel('Total Log-Likelihood')
axes[1,0].set_title('GMM | Log-Likelihood Trend')
axes[1,0].set_xticks(ks)

# (1,1) PCA 2D scatter: K-Means k=3
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_e)
colors_km = ['steelblue','darkorange','seagreen']
for c in range(3):
    mask = km_labels_all[3] == c
    axes[1,1].scatter(X_2d[mask,0], X_2d[mask,1],
                      s=4, alpha=0.3, color=colors_km[c], label=f'Cluster {c}')
axes[1,1].set_title('K-Means k=3 | PCA 2D')
axes[1,1].set_xlabel('PC1')
axes[1,1].set_ylabel('PC2')
axes[1,1].legend(markerscale=3)

# (1,2) PCA 2D scatter: GMM k=3
for c in range(3):
    mask = gm3.predict(X_e) == c
    axes[1,2].scatter(X_2d[mask,0], X_2d[mask,1],
                      s=4, alpha=0.3, color=colors_km[c], label=f'Comp {c}')
axes[1,2].set_title('GMM k=3 | PCA 2D')
axes[1,2].set_xlabel('PC1')
axes[1,2].set_ylabel('PC2')
axes[1,2].legend(markerscale=3)

plt.tight_layout()
plt.show()

# Comparison table
comp_table = pd.DataFrame({
    'k': ks,
    'KM Inertia':    [f'{v:.0f}' for v in km_inertia],
    'KM Silhouette': [f'{v:.4f}' for v in km_silhouette],
    'GMM BIC':       [f'{v:.0f}' for v in gmm_bic],
    'GMM AIC':       [f'{v:.0f}' for v in gmm_aic],
    'GMM LogLik':    [f'{v:.0f}' for v in gmm_loglik],
})
print('K-Means vs GMM Summary Table (k = 2-8)')
print(comp_table.to_string(index=False))

### E. Interpretation

**Feature space:** 47-feature `X_val_pp` reduced to 10 PCA components (59.2% variance retained).

**K-Means:** sharpest SSE drop at k=3; silhouette peaks at k=3 (0.263) and k=6 (0.296), both moderate. Cluster sizes at k=3 are uneven (2782 / 2151 / 67).

**GMM:** BIC drops sharply at k=3 (81,383 → 14,178); 99.64% of points have posterior confidence > 0.9, mean entropy = 0.004 — near-hard assignments despite soft model. Component weights skewed: [0.089, 0.053, 0.858].

**K-Means vs GMM:** both agree on k=3 structure; K-Means uses hard Voronoi boundaries, GMM soft elliptical. GMM additionally identifies 0.36% ambiguous boundary points.

**Clusters vs labels:** partial overlap only — K-Means Cluster 1 enriched for High (51%), Cluster 0 skews Standard (58%), but no clean separation. Unsupervised geometry broadly reflects risk level; supervised learning is necessary for reliable classification.

---
## F Final Model Choice and Hidden-Test Export

Best config from validation evidence: **B3 (Log + Interactions + Target Encoding)** (Val Macro-F1 = 0.8913).  
Retrain on combined train + validation data, then generate `test_predictions.csv`.

In [ ]:
# Retrain on train+val, export test CSV

# Step 1: combine train + val (pre-cleaned)
X_full_raw = pd.concat([train_clean, val_clean], ignore_index=True)
y_full     = X_full_raw[TARGET]
X_full     = X_full_raw.drop(columns=[TARGET])

y_full_enc     = label_enc.transform(y_full)
y_full_ordinal = y_full.map(RISK_ORDINAL).values

# Step 2: re-fit feature engineering
ste_full = SmoothTargetEncoder(cols=CAT_COLS, smoothing=10).fit(X_full, y_full_ordinal)

X_full1 = add_log_transforms(X_full)
X_full2 = add_interactions(X_full1)
X_full3 = ste_full.transform(X_full2)

# Step 3: re-fit preprocessor (B3: all numeric, no OHE)
full_num_cols = NUM_COLS + log_new + INT_COLS + CAT_COLS
pp_full = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), full_num_cols)
], remainder='drop')

X_full_pp = pp_full.fit_transform(X_full3)
print(f'Full train+val feature matrix: {X_full_pp.shape}')

# Step 4: retrain XGBoost (Optuna best params)
xgb_final = XGBClassifier(**best_params)
xgb_final.fit(X_full_pp, y_full_enc)
print('Retrain complete.')

# Step 5: transform test set
X_te1 = add_log_transforms(test_clean)
X_te2 = add_interactions(X_te1)
X_te3 = ste_full.transform(X_te2)
X_te_pp = pp_full.transform(X_te3)

y_te_enc = xgb_final.predict(X_te_pp)
y_te     = label_enc.inverse_transform(y_te_enc)

print(f'\nTest prediction distribution:')
for cls, cnt in zip(*np.unique(y_te, return_counts=True)):
    print(f'  {cls:10s}: {cnt:5d}  ({cnt/len(y_te)*100:.1f}%)')

# Step 6: assemble output
result = pd.DataFrame({
    'applicant_id': test_admin['applicant_id'].values,
    'customer_key': test_admin['customer_key'].values,
    'premium_risk': y_te
})

assert len(result) == len(test)
assert set(result['premium_risk'].unique()).issubset({'Low', 'Standard', 'High'})

OUT_PATH = 'test_predictions.csv'
result.to_csv(OUT_PATH, index=False)
print(f'\nSaved: {OUT_PATH}  ({len(result)} rows)')
print(result.head(5).to_string(index=False))